# 📈 Probabilistic & Uncertainty Interval Evaluation Report

Evaluates Prediction Interval Coverage Probability (PICP) & Mean Prediction Interval Width (MPIW) across all 8 drugs.


In [1]:
import os, glob, pandas as pd, numpy as np

BASE_DIR = os.getcwd()
DRUG_FOLDERS = ['m01ab_models', 'm01ae_models', 'n02ba_models', 'n02be_models', 
                'n05b_models', 'n05c_models', 'r03_models', 'r06_models']

results = []
for folder_name in DRUG_FOLDERS:
    folder_path = os.path.join(BASE_DIR, folder_name)
    csv_files = glob.glob(os.path.join(folder_path, '*_supply_chain_plan.csv'))
    if not csv_files:
        continue
    csv_path = csv_files[0]
    df = pd.read_csv(csv_path)
    drug_code = folder_name.split('_')[0].upper()
    
    df_2019 = df[df['Date'].str.startswith('2019')].dropna(subset=['Actual Sales'])
    y_true = df_2019['Actual Sales'].values
    p10    = df_2019['Lean Lower Bound (P10)'].values
    p90    = df_2019['Upper Target Stock (P90)'].values
    
    picp = np.mean((y_true >= p10) & (y_true <= p90)) * 100
    p90_coverage = np.mean(y_true <= p90) * 100
    mpiw = np.mean(p90 - p10)
    
    results.append({
        'Drug Category': drug_code,
        'PICP (%) [P10-P90 Coverage]': round(picp, 2),
        'P90 Service Level (%)': round(p90_coverage, 2),
        'MPIW (Pack Units)': round(mpiw, 2),
        'Target PICP Interval': '80% Band [P10, P90]'
    })

eval_df = pd.DataFrame(results)
avg_row = {
    'Drug Category': 'PORTFOLIO AVG',
    'PICP (%) [P10-P90 Coverage]': round(eval_df['PICP (%) [P10-P90 Coverage]'].mean(), 2),
    'P90 Service Level (%)': round(eval_df['P90 Service Level (%)'].mean(), 2),
    'MPIW (Pack Units)': round(eval_df['MPIW (Pack Units)'].mean(), 2),
    'Target PICP Interval': '80% Nominal Target'
}
eval_df_final = pd.concat([eval_df, pd.DataFrame([avg_row])], ignore_index=True)

display(eval_df_final)
output_csv = os.path.join(BASE_DIR, 'probabilistic_forecasting_eval.csv')
eval_df_final.to_csv(output_csv, index=False)
